# House Price Prediction

## Notebook 4 — Model Training

### Objectives

- Load processed datasets
- Train multiple regression models
- Compare model performance
- Save trained models

Imports

In [16]:
import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

Project Paths

In [17]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"

MODELS.mkdir(parents=True, exist_ok=True)

Load Data

In [18]:
X_train = pd.read_csv(PROCESSED / "X_train.csv")
X_valid = pd.read_csv(PROCESSED / "X_valid.csv")

y_train = pd.read_csv(PROCESSED / "y_train.csv").squeeze()
y_valid = pd.read_csv(PROCESSED / "y_valid.csv").squeeze()

print(X_train.shape)
print(X_valid.shape)

(1168, 245)
(292, 245)


Models

In [19]:
models = {

    "Linear Regression":
        LinearRegression(),

    "Decision Tree":
        DecisionTreeRegressor(
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        )
}

Evaluation Function

In [20]:
def evaluate_model(name, model):

    start = time.time()

    model.fit(X_train, y_train)

    predictions = model.predict(X_valid)

    runtime = time.time() - start

    rmse = np.sqrt(
        mean_squared_error(
            y_valid,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_valid,
        predictions
    )

    r2 = r2_score(
        y_valid,
        predictions
    )

    return {

        "Model": name,

        "RMSE": rmse,

        "MAE": mae,

        "R2": r2,

        "Time (s)": runtime,

        "Estimator": model
    }

Train Models

In [21]:
results = []

for name, model in models.items():

    print(f"Training {name}...")

    results.append(
        evaluate_model(
            name,
            model
        )
    )

Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...


Results Table

In [22]:
results_df = pd.DataFrame(results)

results_df = results_df.drop(
    columns="Estimator"
)

results_df.sort_values(
    by="RMSE",
    inplace=True
)

results_df

,Model,RMSE,MAE,R2,Time (s)
3,Gradient Boosting,28067.403691,17458.613829,0.897295,1.561311
4,XGBoost,28514.238969,17760.125000,0.893999,0.950948
2,Random Forest,28631.231542,17577.349007,0.893127,14.171498
1,Decision Tree,39609.713326,26107.750000,0.795455,0.093729
0,Linear Regression,51405.094253,20232.175888,0.655493,0.063545


Best Model

In [23]:
best_result = min(
    results,
    key=lambda x: x["RMSE"]
)

best_model = best_result["Estimator"]

print("Best Model:", best_result["Model"])
print("RMSE:", best_result["RMSE"])

Best Model: Gradient Boosting
RMSE: 28067.403691339066


Save Models

In [24]:
for item in results:

    filename = (
        item["Model"]
        .lower()
        .replace(" ", "_")
        + ".pkl"
    )

    joblib.dump(
        item["Estimator"],
        MODELS / filename
    )

print("All models saved.")

All models saved.


Save Best Model

In [25]:
joblib.dump(
    best_model,
    MODELS / "best_model.pkl"
)

print("Best model saved.")

Best model saved.


Predictions

In [26]:
predictions = best_model.predict(X_valid)

predictions[:10]

array([147525.44173365, 341041.88958443, 123036.76230339, 147051.34036335,
       329793.8881688 ,  81317.00788251, 227635.82678569, 133872.56152363,
        79349.9234983 , 129009.75266768])

Actual vs Predicted

In [27]:
comparison = pd.DataFrame({

    "Actual": y_valid,

    "Predicted": predictions

})

comparison.head(20)

,Actual,Predicted
0,154500,147525.441734
1,325000,341041.889584
2,115000,123036.762303
3,159000,147051.340363
4,315500,329793.888169
5,75500,81317.007883
6,311500,227635.826786
7,146000,133872.561524
8,84500,79349.923498
9,135500,129009.752668


Save Results

In [28]:
results_df.to_csv(
    ROOT / "reports" / "model_results.csv",
    index=False
)

print("Results saved.")

Results saved.


Summary

In [29]:
print(results_df)

               Model          RMSE           MAE        R2   Time (s)
3  Gradient Boosting  28067.403691  17458.613829  0.897295   1.561311
4            XGBoost  28514.238969  17760.125000  0.893999   0.950948
2      Random Forest  28631.231542  17577.349007  0.893127  14.171498
1      Decision Tree  39609.713326  26107.750000  0.795455   0.093729
0  Linear Regression  51405.094253  20232.175888  0.655493   0.063545
